# **📈 Notebook 2 — Clasificador SVC de Tendencia Bursátil**
## iDeSo · UNMSM · FISI


---

**Objetivo:** Implementar el pipeline completo del Support Vector Classifier (SVC) para clasificación
binaria de tendencias bursátiles (BUY / SELL) sobre 5 activos financieros mineros con operaciones en Perú.

**Activos del estudio:**
| Ticker | Empresa | Bolsa |
|--------|---------|-------|
| FSM | Fortuna Silver Mines Inc. | NYSE |
| VOLCABC1.LM | Volcan Compañía Minera S.A.A. | BVL |
| ABX.TO | Barrick Gold Corporation | TSX |
| BVN | Compañía de Minas Buenaventura | NYSE |
| BHP | BHP Billiton Limited | NYSE |

**Salida:** Archivo `datos_svc.json` con estructura compatible con `ernesto_investing_svc_clasificador.html`

## Módulo 0 — Configuración de Activos

In [1]:
# ═══════════════════════════════════════════════════════════════
# MÓDULO 0 — CONFIGURACIÓN DE ACTIVOS DEL ESTUDIO
# Tickers de empresas mineras con operaciones en Perú
# ═══════════════════════════════════════════════════════════════

# Semilla fija para reproducibilidad de resultados
SEED = 42

# Lista de tickers
TICKERS = [
    'FSM',         # Fortuna Silver Mines — NYSE
    'VOLCABC1.LM', # Volcan Compañía Minera — BVL
    'ABX.TO',      # Barrick Gold Corporation — TSX
    'BVN',         # Compañía de Minas Buenaventura — NYSE
    'BHP'          # BHP Billiton Limited — NYSE
]

# Nombres completos para el JSON de salida
NOMBRES_EMPRESAS = {
    'FSM':         'Fortuna Silver Mines Inc.',
    'VOLCABC1.LM': 'Volcan Compañía Minera S.A.A.',
    'ABX.TO':      'Barrick Gold Corporation',
    'BVN':         'Compañía de Minas Buenaventura S.A.A.',
    'BHP':         'BHP Billiton Limited'
}

# Periodo de descarga: últimos 2 años
PERIODO = '2y'

# Proporción de partición entrenamiento / prueba
TRAIN_RATIO = 0.80

print('✅ Configuración cargada correctamente.')
print(f'   Tickers: {TICKERS}')
print(f'   Periodo: {PERIODO} | Semilla: {SEED} | Train/Test: {int(TRAIN_RATIO*100)}/{int((1-TRAIN_RATIO)*100)}')

✅ Configuración cargada correctamente.
   Tickers: ['FSM', 'VOLCABC1.LM', 'ABX.TO', 'BVN', 'BHP']
   Periodo: 2y | Semilla: 42 | Train/Test: 80/19


## Módulo 1 — Instalación e Importación de Librerías

In [2]:
# ═══════════════════════════════════════════════════════════════
# MÓDULO 1 — INSTALACIÓN DE LIBRERÍAS
# Ejecutar solo si las librerías no están instaladas en el entorno
# ═══════════════════════════════════════════════════════════════

!pip install yfinance ta scikit-learn pandas numpy --quiet
print('✅ Librerías instaladas correctamente.')

  Preparing metadata (setup.py) ... done
✅ Librerías instaladas correctamente.


In [3]:
# ═══════════════════════════════════════════════════════════════
# MÓDULO 1 — IMPORTACIÓN DE LIBRERÍAS
# ═══════════════════════════════════════════════════════════════

import warnings
warnings.filterwarnings('ignore')

# Librerías de datos y cálculo numérico
import numpy as np
import pandas as pd
import json
from datetime import datetime

# Descarga de datos financieros reales
import yfinance as yf

# Indicadores técnicos financieros
import ta
from ta.trend import SMAIndicator, EMAIndicator, MACD
from ta.momentum import RSIIndicator, StochasticOscillator
from ta.volatility import BollingerBands, AverageTrueRange

# Machine Learning — scikit-learn
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

# Reproducibilidad
np.random.seed(SEED)

print('✅ Todas las librerías importadas correctamente.')
print(f'   numpy {np.__version__} | pandas {pd.__version__} | yfinance {yf.__version__}')

✅ Todas las librerías importadas correctamente.
   numpy 2.0.2 | pandas 2.2.2 | yfinance 0.2.66


## Módulo 2 — Descarga de Datos OHLCV con yfinance

In [4]:
# ═══════════════════════════════════════════════════════════════
# MÓDULO 2 — DESCARGA DE DATOS DE MERCADO CON YFINANCE
# Datos OHLCV reales de Yahoo Finance para los 5 tickers
# ═══════════════════════════════════════════════════════════════

def descargar_datos_ohlcv(tickers: list, periodo: str) -> dict:
    """
    Descarga datos OHLCV reales de Yahoo Finance para una lista de tickers.

    Parámetros:
        tickers  : Lista de símbolos bursátiles
        periodo  : Ventana temporal (ej. '2y' = 2 años)

    Retorna:
        Diccionario { ticker: DataFrame } con los datos OHLCV descargados
    """
    datos_crudos = {}  # Almacén de datos por ticker

    for ticker in tickers:
        print(f'   ⬇️  Descargando {ticker}...')
        try:
            # Descarga real desde Yahoo Finance — nunca datos inventados
            df = yf.download(ticker, period=periodo, auto_adjust=True, progress=False)

            if df.empty:
                print(f'   ⚠️  {ticker}: sin datos disponibles. Se omitirá.')
                continue

            # Aplanar columnas MultiIndex si las hay (ocurre con auto_adjust)
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)

            # Eliminar filas con valores nulos en columnas clave
            df.dropna(subset=['Close', 'Open', 'High', 'Low', 'Volume'], inplace=True)

            # Guardar solo las columnas OHLCV necesarias
            df = df[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
            datos_crudos[ticker] = df

            print(f'      ✅ {ticker}: {len(df)} días | '
                  f'{df.index[0].date()} → {df.index[-1].date()} | '
                  f'Último cierre: {df["Close"].iloc[-1]:.4f}')

        except Exception as e:
            print(f'   ❌ {ticker}: error al descargar — {e}')

    return datos_crudos


print('🌐 Iniciando descarga de datos reales desde Yahoo Finance...')
datos_mercado = descargar_datos_ohlcv(TICKERS, PERIODO)
print(f'\n📦 Tickers con datos descargados: {list(datos_mercado.keys())}')

🌐 Iniciando descarga de datos reales desde Yahoo Finance...
   ⬇️  Descargando FSM...
      ✅ FSM: 501 días | 2024-06-20 → 2026-06-18 | Último cierre: 9.2600
   ⬇️  Descargando VOLCABC1.LM...
      ✅ VOLCABC1.LM: 494 días | 2024-06-20 → 2026-06-18 | Último cierre: 0.8890
   ⬇️  Descargando ABX.TO...
      ✅ ABX.TO: 503 días | 2024-06-19 → 2026-06-19 | Último cierre: 56.1900
   ⬇️  Descargando BVN...
      ✅ BVN: 501 días | 2024-06-20 → 2026-06-18 | Último cierre: 32.5800
   ⬇️  Descargando BHP...
      ✅ BHP: 501 días | 2024-06-20 → 2026-06-18 | Último cierre: 87.8700

📦 Tickers con datos descargados: ['FSM', 'VOLCABC1.LM', 'ABX.TO', 'BVN', 'BHP']


## Módulo 3 — Ingeniería de Características

In [5]:
# ═══════════════════════════════════════════════════════════════
# MÓDULO 3 — INGENIERÍA DE CARACTERÍSTICAS
# Cálculo de indicadores técnicos y variables de entrada al SVC
# ═══════════════════════════════════════════════════════════════

def calcular_caracteristicas(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calcula el conjunto completo de características técnicas para el SVC.

    Indicadores calculados:
      - Retornos diarios y lag de retornos
      - SMA-20, SMA-50 (Medias Móviles Simples)
      - EMA-12, EMA-26 (Medias Móviles Exponenciales)
      - RSI-14 (Relative Strength Index)
      - MACD, Señal MACD, Histograma MACD
      - Bandas de Bollinger (banda superior, media, inferior, ancho)
      - ATR-14 (Average True Range — volatilidad)
      - Oscilador Estocástico %K y %D
      - Momentum (retorno a 10 días)
      - Volatilidad histórica a 20 días

    Parámetros:
        df : DataFrame con columnas OHLCV

    Retorna:
        DataFrame enriquecido con todas las características
    """
    datos = df.copy()

    # ── Retornos diarios y retornos con rezago ──
    datos['retorno']      = datos['Close'].pct_change()           # Retorno t
    datos['retorno_lag1'] = datos['retorno'].shift(1)             # Retorno t-1
    datos['retorno_lag2'] = datos['retorno'].shift(2)             # Retorno t-2
    datos['retorno_lag3'] = datos['retorno'].shift(3)             # Retorno t-3

    # ── SMA: Medias Móviles Simples ──
    datos['sma_20'] = SMAIndicator(close=datos['Close'], window=20).sma_indicator()
    datos['sma_50'] = SMAIndicator(close=datos['Close'], window=50).sma_indicator()

    # ── EMA: Medias Móviles Exponenciales ──
    datos['ema_12'] = EMAIndicator(close=datos['Close'], window=12).ema_indicator()
    datos['ema_26'] = EMAIndicator(close=datos['Close'], window=26).ema_indicator()

    # ── Diferencias SMA/EMA respecto al precio (señales cruzadas) ──
    datos['precio_vs_sma20'] = (datos['Close'] - datos['sma_20']) / datos['sma_20']
    datos['precio_vs_sma50'] = (datos['Close'] - datos['sma_50']) / datos['sma_50']
    datos['sma20_vs_sma50']  = (datos['sma_20'] - datos['sma_50']) / datos['sma_50']
    datos['ema12_vs_ema26']  = (datos['ema_12'] - datos['ema_26']) / datos['ema_26']

    # ── RSI-14: Relative Strength Index ──
    datos['rsi_14'] = RSIIndicator(close=datos['Close'], window=14).rsi()

    # ── MACD: Moving Average Convergence Divergence ──
    macd_obj           = MACD(close=datos['Close'], window_slow=26, window_fast=12, window_sign=9)
    datos['macd']      = macd_obj.macd()
    datos['macd_senal']= macd_obj.macd_signal()
    datos['macd_hist'] = macd_obj.macd_diff()

    # ── Bandas de Bollinger (ventana=20, desviaciones=2) ──
    bb_obj             = BollingerBands(close=datos['Close'], window=20, window_dev=2)
    datos['bb_alto']   = bb_obj.bollinger_hband()
    datos['bb_medio']  = bb_obj.bollinger_mavg()
    datos['bb_bajo']   = bb_obj.bollinger_lband()
    datos['bb_ancho']  = (datos['bb_alto'] - datos['bb_bajo']) / datos['bb_medio']
    datos['bb_posicion'] = (datos['Close'] - datos['bb_bajo']) / (datos['bb_alto'] - datos['bb_bajo'] + 1e-9)

    # ── ATR-14: Average True Range (volatilidad de rango verdadero) ──
    datos['atr_14'] = AverageTrueRange(
        high=datos['High'], low=datos['Low'], close=datos['Close'], window=14
    ).average_true_range()

    # ── Oscilador Estocástico %K y %D ──
    stoch_obj         = StochasticOscillator(
        high=datos['High'], low=datos['Low'], close=datos['Close'], window=14, smooth_window=3
    )
    datos['stoch_k']  = stoch_obj.stoch()
    datos['stoch_d']  = stoch_obj.stoch_signal()

    # ── Momentum a 10 días ──
    datos['momentum_10'] = datos['Close'].pct_change(periods=10)

    # ── Volatilidad histórica a 20 días ──
    datos['volatilidad_20'] = datos['retorno'].rolling(window=20).std() * np.sqrt(252)

    # ── Volumen relativo (ratio vs media móvil del volumen) ──
    datos['volumen_ratio'] = datos['Volume'] / datos['Volume'].rolling(20).mean()

    return datos


# Calcular características para cada ticker
print('⚙️  Calculando indicadores técnicos para todos los tickers...')
datos_con_features = {}

for ticker in datos_mercado:
    datos_con_features[ticker] = calcular_caracteristicas(datos_mercado[ticker])
    cols_features = [c for c in datos_con_features[ticker].columns if c not in ['Open','High','Low','Close','Volume']]
    print(f'   ✅ {ticker}: {len(cols_features)} características calculadas')

print(f'\n📊 Ejemplo de características para {list(datos_con_features.keys())[0]}:')
print(datos_con_features[list(datos_con_features.keys())[0]][cols_features].tail(3).to_string())

⚙️  Calculando indicadores técnicos para todos los tickers...
   ✅ FSM: 27 características calculadas
   ✅ VOLCABC1.LM: 27 características calculadas
   ✅ ABX.TO: 27 características calculadas
   ✅ BVN: 27 características calculadas
   ✅ BHP: 27 características calculadas

📊 Ejemplo de características para FSM:
Price        retorno  retorno_lag1  retorno_lag2  retorno_lag3  sma_20  sma_50    ema_12    ema_26  precio_vs_sma20  precio_vs_sma50  sma20_vs_sma50  ema12_vs_ema26     rsi_14      macd  macd_senal  macd_hist    bb_alto  bb_medio   bb_bajo  bb_ancho  bb_posicion    atr_14    stoch_k    stoch_d  momentum_10  volatilidad_20  volumen_ratio
Date                                                                                                                                                                                                                                                                                                                                              
2026-06-1

## Módulo 4 — Variable Objetivo Binaria (Trend)

In [6]:
# ═══════════════════════════════════════════════════════════════
# MÓDULO 4 — CREACIÓN DE LA VARIABLE OBJETIVO BINARIA
# Trend: 1 = BUY (retorno siguiente positivo)
#        0 = SELL (retorno siguiente negativo o cero)
# ═══════════════════════════════════════════════════════════════

def crear_variable_objetivo(df: pd.DataFrame) -> pd.DataFrame:
    """
    Agrega la variable objetivo binaria 'Trend' al DataFrame.

    La señal se define sobre el retorno del DÍA SIGUIENTE (t+1):
      - Trend = 1 (BUY)  si el precio de cierre del día siguiente > hoy
      - Trend = 0 (SELL) si el precio de cierre del día siguiente <= hoy

    IMPORTANTE: Se usa shift(-1) para mirar hacia adelante, pero en la
    partición train/test solo se usa información pasada (sin data leakage).

    Parámetros:
        df : DataFrame con precios y características

    Retorna:
        DataFrame con columna 'Trend' añadida y última fila eliminada
    """
    datos = df.copy()

    # Retorno del día siguiente: shift(-1) mira el precio futuro
    retorno_siguiente = datos['Close'].shift(-1) / datos['Close'] - 1

    # Clasificación binaria: 1 = BUY, 0 = SELL
    datos['Trend'] = (retorno_siguiente > 0).astype(int)

    # Eliminar la última fila (no tiene retorno siguiente real)
    datos.dropna(inplace=True)

    return datos


# Aplicar a todos los tickers
print('🎯 Creando variable objetivo binaria Trend (BUY=1 / SELL=0)...')
datos_con_objetivo = {}

for ticker in datos_con_features:
    datos_con_objetivo[ticker] = crear_variable_objetivo(datos_con_features[ticker])
    df_t = datos_con_objetivo[ticker]
    n_buy  = df_t['Trend'].sum()
    n_sell = len(df_t) - n_buy
    print(f'   ✅ {ticker}: {len(df_t)} muestras | BUY={n_buy} ({n_buy/len(df_t)*100:.1f}%) | SELL={n_sell} ({n_sell/len(df_t)*100:.1f}%)')

print('\n✅ Variable objetivo generada sin data leakage.')

🎯 Creando variable objetivo binaria Trend (BUY=1 / SELL=0)...
   ✅ FSM: 452 muestras | BUY=234 (51.8%) | SELL=218 (48.2%)
   ✅ VOLCABC1.LM: 445 muestras | BUY=208 (46.7%) | SELL=237 (53.3%)
   ✅ ABX.TO: 454 muestras | BUY=248 (54.6%) | SELL=206 (45.4%)
   ✅ BVN: 452 muestras | BUY=242 (53.5%) | SELL=210 (46.5%)
   ✅ BHP: 452 muestras | BUY=241 (53.3%) | SELL=211 (46.7%)

✅ Variable objetivo generada sin data leakage.


## Módulo 5 — Normalización y Partición Train/Test

In [7]:
# ═══════════════════════════════════════════════════════════════
# MÓDULO 5 — NORMALIZACIÓN CON StandardScaler Y PARTICIÓN TEMPORAL
# Se respeta el orden cronológico (no se hace shuffle)
# ═══════════════════════════════════════════════════════════════

# Columnas de características a usar como entrada del SVC
COLUMNAS_FEATURES = [
    'retorno', 'retorno_lag1', 'retorno_lag2', 'retorno_lag3',
    'precio_vs_sma20', 'precio_vs_sma50', 'sma20_vs_sma50', 'ema12_vs_ema26',
    'rsi_14',
    'macd', 'macd_senal', 'macd_hist',
    'bb_ancho', 'bb_posicion',
    'atr_14',
    'stoch_k', 'stoch_d',
    'momentum_10', 'volatilidad_20', 'volumen_ratio'
]


def preparar_datos_modelo(df: pd.DataFrame, features: list, train_ratio: float):
    """
    Prepara los datos para el modelo SVC:
      1. Selecciona las columnas de características
      2. Elimina filas con NaN (período de calentamiento de indicadores)
      3. Particiona en train/test respetando el orden temporal (sin shuffle)
      4. Normaliza con StandardScaler ajustado SOLO sobre train

    Parámetros:
        df          : DataFrame con características y variable objetivo
        features    : Lista de columnas a usar como features
        train_ratio : Proporción del conjunto de entrenamiento

    Retorna:
        X_train_sc, X_test_sc, y_train, y_test, scaler, fechas_train, fechas_test
    """
    # Filtrar solo filas sin valores nulos en las features
    cols_necesarias = features + ['Trend']
    df_limpio = df[cols_necesarias].dropna().copy()

    X = df_limpio[features].values
    y = df_limpio['Trend'].values
    fechas = df_limpio.index

    # Partición temporal (sin shuffle — respeta el orden cronológico)
    n_train = int(len(X) * train_ratio)
    X_train, X_test = X[:n_train], X[n_train:]
    y_train, y_test = y[:n_train], y[n_train:]
    fechas_train    = fechas[:n_train]
    fechas_test     = fechas[n_train:]

    # Normalización: el scaler aprende SOLO con train (evita data leakage)
    scaler    = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc  = scaler.transform(X_test)    # Solo transform en test

    return X_train_sc, X_test_sc, y_train, y_test, scaler, fechas_train, fechas_test, df_limpio


# Preparar datos para cada ticker
print('⚖️  Normalizando y particionando datos (sin shuffle — orden cronológico)...')
datos_preparados = {}

for ticker in datos_con_objetivo:
    resultado = preparar_datos_modelo(
        datos_con_objetivo[ticker], COLUMNAS_FEATURES, TRAIN_RATIO
    )
    datos_preparados[ticker] = {
        'X_train': resultado[0],
        'X_test':  resultado[1],
        'y_train': resultado[2],
        'y_test':  resultado[3],
        'scaler':  resultado[4],
        'fechas_train': resultado[5],
        'fechas_test':  resultado[6],
        'df_limpio':    resultado[7]
    }
    print(f'   ✅ {ticker}: Train={len(resultado[2])} | Test={len(resultado[3])} | Features={resultado[0].shape[1]}')

print(f'\n📐 Dimensiones de ejemplo — {list(datos_preparados.keys())[0]}:')
ticker_ej = list(datos_preparados.keys())[0]
print(f'   X_train: {datos_preparados[ticker_ej]["X_train"].shape}')
print(f'   X_test:  {datos_preparados[ticker_ej]["X_test"].shape}')

⚖️  Normalizando y particionando datos (sin shuffle — orden cronológico)...
   ✅ FSM: Train=361 | Test=91 | Features=20
   ✅ VOLCABC1.LM: Train=356 | Test=89 | Features=20
   ✅ ABX.TO: Train=363 | Test=91 | Features=20
   ✅ BVN: Train=361 | Test=91 | Features=20
   ✅ BHP: Train=361 | Test=91 | Features=20

📐 Dimensiones de ejemplo — FSM:
   X_train: (361, 20)
   X_test:  (91, 20)


## Módulo 6 — Entrenamiento con GridSearchCV

In [10]:
# ═══════════════════════════════════════════════════════════════
# MÓDULO 6 — ENTRENAMIENTO DEL SVC CON GridSearchCV
# Optimización sobre kernels, C y gamma usando validación cruzada
# temporal (TimeSeriesSplit) para no romper el orden cronológico
# ═══════════════════════════════════════════════════════════════

# Grilla de hiperparámetros a explorar
PARAM_GRID = {
    'kernel': ['linear', 'rbf', 'poly', 'sigmoid'],
    'C':      [0.1, 1, 10, 100],
    'gamma':  ['scale', 'auto']
}

# Validación cruzada temporal (mantiene el orden del tiempo)
cv_temporal = TimeSeriesSplit(n_splits=5)


def entrenar_svc_con_gridsearch(X_train, y_train, param_grid: dict, cv):
    """
    Entrena un SVC con búsqueda exhaustiva de hiperparámetros.

    Parámetros:
        X_train    : Matriz de características normalizada
        y_train    : Vector de etiquetas de entrenamiento
        param_grid : Diccionario con la grilla de parámetros
        cv         : Estrategia de validación cruzada

    Retorna:
        Mejor estimador encontrado por GridSearchCV
    """
    svc_base = SVC(
        probability=True,   # Necesario para obtener probabilidades de confianza
        random_state=SEED,
        max_iter=2000       # Límite de iteraciones para convergencia
    )

    grid_search = GridSearchCV(
        estimator=svc_base,
        param_grid=param_grid,
        cv=cv,
        scoring='f1',          # Optimizar por F1 (balanceado para clases desiguales)
        n_jobs=-1,             # Usar todos los núcleos disponibles
        verbose=0,
        refit=True             # Re-entrenar con mejores parámetros sobre todo el train
    )

    grid_search.fit(X_train, y_train)
    return grid_search


# Entrenar SVC para cada ticker
print('🤖 Entrenando SVC con GridSearchCV para cada ticker...')
print(f'   Grilla: kernels={PARAM_GRID["kernel"]} | C={PARAM_GRID["C"]} | gamma={PARAM_GRID["gamma"]}')
print(f'   Validación cruzada temporal: {cv_temporal.n_splits} splits\n')

modelos_svc = {}

for ticker in datos_preparados:
    print(f'   🔄 Entrenando {ticker}...')
    X_train = datos_preparados[ticker]['X_train']
    y_train = datos_preparados[ticker]['y_train']

    grid = entrenar_svc_con_gridsearch(X_train, y_train, PARAM_GRID, cv_temporal)
    modelos_svc[ticker] = grid

    mejores = grid.best_params_
    print(f'      ✅ {ticker} — Mejor: kernel={mejores["kernel"]} | C={mejores["C"]} | gamma={mejores["gamma"]} | CV-F1={grid.best_score_:.4f}')

print('\n✅ Entrenamiento completo para todos los tickers.')

🤖 Entrenando SVC con GridSearchCV para cada ticker...
   Grilla: kernels=['linear', 'rbf', 'poly', 'sigmoid'] | C=[0.1, 1, 10, 100] | gamma=['scale', 'auto']
   Validación cruzada temporal: 5 splits

   🔄 Entrenando FSM...
      ✅ FSM — Mejor: kernel=rbf | C=0.1 | gamma=scale | CV-F1=0.5447
   🔄 Entrenando VOLCABC1.LM...
      ✅ VOLCABC1.LM — Mejor: kernel=rbf | C=10 | gamma=auto | CV-F1=0.5181
   🔄 Entrenando ABX.TO...
      ✅ ABX.TO — Mejor: kernel=sigmoid | C=1 | gamma=scale | CV-F1=0.6130
   🔄 Entrenando BVN...
      ✅ BVN — Mejor: kernel=sigmoid | C=10 | gamma=scale | CV-F1=0.5868
   🔄 Entrenando BHP...
      ✅ BHP — Mejor: kernel=rbf | C=0.1 | gamma=scale | CV-F1=0.6688

✅ Entrenamiento completo para todos los tickers.


## Módulo 7 — Evaluación de Métricas

In [11]:
# ═══════════════════════════════════════════════════════════════
# MÓDULO 7 — EVALUACIÓN DEL CLASIFICADOR SVC
# Métricas: accuracy, precision, recall, F1, matriz de confusión
# Predicciones por fecha para alimentar el gráfico del frontend
# ═══════════════════════════════════════════════════════════════

def evaluar_modelo(
    grid_search,
    X_test, y_test,
    X_train, y_train,
    fechas_test, fechas_train,
    df_limpio, ticker: str
) -> dict:
    """
    Evalúa el mejor SVC encontrado y calcula todas las métricas.

    Retorna un diccionario con la estructura que el frontend HTML espera.
    """
    modelo = grid_search.best_estimator_
    mejores_params = grid_search.best_params_

    # ── Predicciones en TEST ──
    y_pred      = modelo.predict(X_test)
    y_prob      = modelo.predict_proba(X_test)[:, 1]   # Probabilidad de BUY

    # ── Predicciones en TRAIN (para graficar señales históricas completas) ──
    y_pred_train = modelo.predict(X_train)
    y_prob_train = modelo.predict_proba(X_train)[:, 1]

    # ── Métricas en conjunto de TEST ──
    accuracy  = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall    = recall_score(y_test, y_pred, zero_division=0)
    f1        = f1_score(y_test, y_pred, zero_division=0)
    cm        = confusion_matrix(y_test, y_pred)

    # ── Descomposición de la matriz de confusión ──
    tn, fp, fn, tp = cm.ravel()

    # ── Señal actual (última predicción disponible) ──
    ultima_prob  = float(y_prob[-1])
    ultima_senal = 'BUY' if ultima_prob >= 0.5 else 'SELL'

    # ── Construir historial de señales COMPLETO (train + test) para el gráfico ──
    # Se usan los últimos 180 días para mantener el gráfico manejable
    todas_fechas  = list(fechas_train.strftime('%Y-%m-%d')) + list(fechas_test.strftime('%Y-%m-%d'))
    todas_senal   = ['BUY' if s == 1 else 'SELL' for s in list(y_pred_train) + list(y_pred)]
    todos_precios = df_limpio.loc[list(fechas_train) + list(fechas_test), 'Close'].round(4).tolist()

    # Limitar a los últimos 180 días de datos
    N_MOSTRAR = 180
    todas_fechas  = todas_fechas[-N_MOSTRAR:]
    todas_senal   = todas_senal[-N_MOSTRAR:]
    todos_precios = todos_precios[-N_MOSTRAR:]

    # ── Distribución de clases en test ──
    n_buy_test  = int(np.sum(y_pred == 1))
    n_sell_test = int(np.sum(y_pred == 0))

    print(f'\n   📊 {ticker} — Resultados en TEST:')
    print(f'      Accuracy:  {accuracy:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}')
    print(f'      Matriz: TP={tp} | FP={fp} | FN={fn} | TN={tn}')
    print(f'      Señal actual: {ultima_senal} (confianza: {ultima_prob:.2f})')
    print(f'      Mejor kernel: {mejores_params["kernel"]} | C={mejores_params["C"]} | gamma={mejores_params["gamma"]}')

    # ── Retornar estructura compatible con el frontend HTML ──
    return {
        # Señal actual y confianza (para el semáforo)
        'senal':  ultima_senal,
        'conf':   round(ultima_prob, 4),

        # Historial de fechas, precios y señales (para el gráfico de línea)
        'fechas':  todas_fechas,
        'precios': [round(p, 4) for p in todos_precios],
        'senales': todas_senal,

        # Métricas del clasificador (para el panel de métricas)
        'metricas': {
            'accuracy':  round(float(accuracy),  4),
            'precision': round(float(precision), 4),
            'recall':    round(float(recall),    4),
            'f1':        round(float(f1),        4)
        },

        # Matriz de confusión 2x2 [[TP, FP], [FN, TN]]
        'matriz': [[int(tp), int(fp)], [int(fn), int(tn)]],

        # Distribución de clases en el test (para el gráfico de barras)
        'clases': {'buy': n_buy_test, 'sell': n_sell_test},

        # Mejor configuración encontrada por GridSearchCV
        'kernel': {
            'tipo':  mejores_params['kernel'],
            'C':     mejores_params['C'],
            'gamma': mejores_params['gamma'],
            'cv_f1_score': round(float(grid_search.best_score_), 4)
        }
    }


# Evaluar todos los tickers
print('📈 Evaluando modelos SVC en el conjunto de TEST...')
resultados_svc = {}

# Temporarily modify preparar_datos_modelo to include 'Close' in df_limpio
def preparar_datos_modelo_fixed(df: pd.DataFrame, features: list, train_ratio: float):
    cols_necesarias = features + ['Trend', 'Close'] # Add 'Close' here
    df_limpio = df[cols_necesarias].dropna().copy()

    X = df_limpio[features].values
    y = df_limpio['Trend'].values
    fechas = df_limpio.index

    n_train = int(len(X) * train_ratio)
    X_train, X_test = X[:n_train], X[n_train:]
    y_train, y_test = y[:n_train], y[n_train:]
    fechas_train    = fechas[:n_train]
    fechas_test     = fechas[n_train:]

    scaler    = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc  = scaler.transform(X_test)

    return X_train_sc, X_test_sc, y_train, y_test, scaler, fechas_train, fechas_test, df_limpio

# Re-prepare data with the fix
print('⚖️  Re-normalizando y particionando datos con la corrección...')
datos_preparados = {}
for ticker in datos_con_objetivo:
    resultado = preparar_datos_modelo_fixed(
        datos_con_objetivo[ticker], COLUMNAS_FEATURES, TRAIN_RATIO
    )
    datos_preparados[ticker] = {
        'X_train': resultado[0],
        'X_test':  resultado[1],
        'y_train': resultado[2],
        'y_test':  resultado[3],
        'scaler':  resultado[4],
        'fechas_train': resultado[5],
        'fechas_test':  resultado[6],
        'df_limpio':    resultado[7]
    }
    print(f'   ✅ {ticker}: Train={len(resultado[2])} | Test={len(resultado[3])} | Features={resultado[0].shape[1]}')

for ticker in modelos_svc:
    resultados_svc[ticker] = evaluar_modelo(
        grid_search  = modelos_svc[ticker],
        X_test       = datos_preparados[ticker]['X_test'],
        y_test       = datos_preparados[ticker]['y_test'],
        X_train      = datos_preparados[ticker]['X_train'],
        y_train      = datos_preparados[ticker]['y_train'],
        fechas_test  = datos_preparados[ticker]['fechas_test'],
        fechas_train = datos_preparados[ticker]['fechas_train'],
        df_limpio    = datos_preparados[ticker]['df_limpio'],
        ticker       = ticker
    )

print('\n✅ Evaluación completa para todos los tickers.')

📈 Evaluando modelos SVC en el conjunto de TEST...
⚖️  Re-normalizando y particionando datos con la corrección...
   ✅ FSM: Train=361 | Test=91 | Features=20
   ✅ VOLCABC1.LM: Train=356 | Test=89 | Features=20
   ✅ ABX.TO: Train=363 | Test=91 | Features=20
   ✅ BVN: Train=361 | Test=91 | Features=20
   ✅ BHP: Train=361 | Test=91 | Features=20

   📊 FSM — Resultados en TEST:
      Accuracy:  0.4725 | Precision: 0.4725 | Recall: 1.0000 | F1: 0.6418
      Matriz: TP=43 | FP=48 | FN=0 | TN=0
      Señal actual: BUY (confianza: 0.51)
      Mejor kernel: rbf | C=0.1 | gamma=scale

   📊 VOLCABC1.LM — Resultados en TEST:
      Accuracy:  0.5393 | Precision: 0.4878 | Recall: 0.5000 | F1: 0.4938
      Matriz: TP=20 | FP=21 | FN=20 | TN=28
      Señal actual: BUY (confianza: 0.58)
      Mejor kernel: rbf | C=10 | gamma=auto

   📊 ABX.TO — Resultados en TEST:
      Accuracy:  0.4286 | Precision: 0.4444 | Recall: 0.7273 | F1: 0.5517
      Matriz: TP=32 | FP=40 | FN=12 | TN=7
      Señal actual: BUY 

## Módulo 8 — Exportación a JSON (Contrato de Datos)

In [12]:
# ═══════════════════════════════════════════════════════════════
# MÓDULO 8 — EXPORTACIÓN A JSON
# Genera datos_svc.json compatible con ernesto_investing_svc_clasificador.html
# Este archivo es el "contrato de datos" entre el backend (Python)
# y el frontend (HTML/JS)
# ═══════════════════════════════════════════════════════════════

# Construir el objeto JSON final con todos los tickers
json_salida = {
    # Metadatos del sistema
    'meta': {
        'sistema':     'Ernesto Investing AI — Clasificador SVC',
        'version':     '1.0',
        'generado_en': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'modelo':      'Support Vector Classifier (SVC)',
        'framework':   'scikit-learn',
        'periodo':     PERIODO,
        'features':    COLUMNAS_FEATURES,
        'train_ratio': TRAIN_RATIO,
        'grid_params': PARAM_GRID,
        'seed':        SEED,
        'curso':       'iDeSo — UNMSM — FISI',
        'profesor':    'Prof. Mg. Ing. Ernesto D. Cancho-Rodríguez, MBA (GWU)'
    },

    # Datos por ticker — estructura que el frontend espera
    'tickers': {}
}

# Agregar resultados de cada ticker
for ticker in resultados_svc:
    json_salida['tickers'][ticker] = {
        # Información de la empresa
        'nombre': NOMBRES_EMPRESAS.get(ticker, ticker),
        'ticker': ticker,

        # Todos los campos que el HTML usa para renderizar
        **resultados_svc[ticker]
    }

# Nombre del archivo de salida
ARCHIVO_SALIDA = 'datos_svc.json'

# Guardar el JSON en disco
with open(ARCHIVO_SALIDA, 'w', encoding='utf-8') as f:
    json.dump(json_salida, f, ensure_ascii=False, indent=2)

print(f'💾 Archivo exportado: {ARCHIVO_SALIDA}')
print(f'   Tickers incluidos: {list(json_salida["tickers"].keys())}')
print(f'   Generado en: {json_salida["meta"]["generado_en"]}')
print()

# ── Vista previa del JSON por ticker ──
print('📋 Vista previa del JSON generado:')
print('=' * 60)
for ticker in json_salida['tickers']:
    t = json_salida['tickers'][ticker]
    print(f'\n  [{ticker}] {t["nombre"]}')
    print(f'    Señal:     {t["senal"]} | Confianza: {t["conf"]*100:.0f}%')
    print(f'    Accuracy:  {t["metricas"]["accuracy"]*100:.1f}%')
    print(f'    F1-Score:  {t["metricas"]["f1"]*100:.1f}%')
    print(f'    Kernel:    {t["kernel"]["tipo"]} | C={t["kernel"]["C"]} | gamma={t["kernel"]["gamma"]}')
    print(f'    Matriz:    TP={t["matriz"][0][0]} FP={t["matriz"][0][1]} FN={t["matriz"][1][0]} TN={t["matriz"][1][1]}')
    print(f'    Puntos graf: {len(t["fechas"])} días')

💾 Archivo exportado: datos_svc.json
   Tickers incluidos: ['FSM', 'VOLCABC1.LM', 'ABX.TO', 'BVN', 'BHP']
   Generado en: 2026-06-20 16:34:53

📋 Vista previa del JSON generado:

  [FSM] Fortuna Silver Mines Inc.
    Señal:     BUY | Confianza: 51%
    Accuracy:  47.2%
    F1-Score:  64.2%
    Kernel:    rbf | C=0.1 | gamma=scale
    Matriz:    TP=43 FP=48 FN=0 TN=0
    Puntos graf: 180 días

  [VOLCABC1.LM] Volcan Compañía Minera S.A.A.
    Señal:     BUY | Confianza: 58%
    Accuracy:  53.9%
    F1-Score:  49.4%
    Kernel:    rbf | C=10 | gamma=auto
    Matriz:    TP=20 FP=21 FN=20 TN=28
    Puntos graf: 180 días

  [ABX.TO] Barrick Gold Corporation
    Señal:     BUY | Confianza: 56%
    Accuracy:  42.9%
    F1-Score:  55.2%
    Kernel:    sigmoid | C=1 | gamma=scale
    Matriz:    TP=32 FP=40 FN=12 TN=7
    Puntos graf: 180 días

  [BVN] Compañía de Minas Buenaventura S.A.A.
    Señal:     BUY | Confianza: 54%
    Accuracy:  50.5%
    F1-Score:  57.1%
    Kernel:    sigmoid | C=10 |

In [13]:
# ═══════════════════════════════════════════════════════════════
# CELDA FINAL — DESCARGA DEL ARCHIVO JSON EN GOOGLE COLAB
# ═══════════════════════════════════════════════════════════════

try:
    # Intentar descarga automática (funciona en Google Colab)
    from google.colab import files
    files.download(ARCHIVO_SALIDA)
    print(f'✅ {ARCHIVO_SALIDA} descargado al computador.')
except ImportError:
    # Si no está en Colab, indicar la ruta local
    import os
    ruta = os.path.abspath(ARCHIVO_SALIDA)
    print(f'✅ Archivo guardado localmente en: {ruta}')

print()
print('═' * 60)
print('  NOTEBOOK 2 COMPLETADO — Clasificador SVC')
print('  Ernesto Investing AI · iDeSo · UNMSM · FISI')
print('═' * 60)
print(f'  📄 Archivo generado : {ARCHIVO_SALIDA}')
print(f'  🌐 Frontend destino : ernesto_investing_svc_clasificador.html')
print(f'  🎯 Tickers procesados: {list(resultados_svc.keys())}')

# Resumen final de accuracy por ticker
print()
print('  📊 Resumen de accuracy por ticker:')
for ticker in resultados_svc:
    acc = resultados_svc[ticker]['metricas']['accuracy']
    f1  = resultados_svc[ticker]['metricas']['f1']
    k   = resultados_svc[ticker]['kernel']['tipo']
    print(f'     {ticker:15s} → Acc={acc*100:.1f}% | F1={f1*100:.1f}% | Kernel={k}')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ datos_svc.json descargado al computador.

════════════════════════════════════════════════════════════
  NOTEBOOK 2 COMPLETADO — Clasificador SVC
  Ernesto Investing AI · iDeSo · UNMSM · FISI
════════════════════════════════════════════════════════════
  📄 Archivo generado : datos_svc.json
  🌐 Frontend destino : ernesto_investing_svc_clasificador.html
  🎯 Tickers procesados: ['FSM', 'VOLCABC1.LM', 'ABX.TO', 'BVN', 'BHP']

  📊 Resumen de accuracy por ticker:
     FSM             → Acc=47.2% | F1=64.2% | Kernel=rbf
     VOLCABC1.LM     → Acc=53.9% | F1=49.4% | Kernel=rbf
     ABX.TO          → Acc=42.9% | F1=55.2% | Kernel=sigmoid
     BVN             → Acc=50.5% | F1=57.1% | Kernel=sigmoid
     BHP             → Acc=56.0% | F1=71.8% | Kernel=rbf
